# EvidGeo: modern models, uncertainty, GenEval 2, and faces/memes

Generates the modern-model extension and scores it. Use an **A100 with High-RAM**. FLUX.2 klein-base fits 40 GB directly; Qwen-Image runs with `--offload` on 40 GB (directly on 80 GB). SD3.5-Medium's reduced design runs on Kaggle 2×T4 (`kaggle/make_kernel.py`, `--device-map balanced`); measured there: ~77 s per guided image, so Kaggle is only practical for small designs.

| Model | Guidance knob | Unguided at | i2i |
|---|---|---|---|
| `sd35-medium` (gated: accept terms on HF, add `HF_TOKEN` to Colab secrets) | `guidance_scale` | 1 | yes |
| `flux2-klein-base` (Apache-2.0) | `guidance_scale` | 1 | no (FLUX.2 `image=` is editing, not SDEdit) |
| `qwen-image` (Apache-2.0) | `true_cfg_scale` + negative prompt | 1 | yes |

Conditions per prompt: a **guidance sweep with the real caption**, a **generic prompt** ("a photo", which reproduces v2's "weak"), and **i2i** (strength 0.75). Seeds are shared across models and modes, so model disagreement at a fixed seed is measurable too.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO = '/content/evidgeocode'
if not os.path.exists(REPO):
    !git clone https://github.com/dpaul0501/evidgeocode.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip -q install -r requirements.txt
MASTER = 'data/manifest_master_N5000_seed42.jsonl'
BASE = '/content/drive/MyDrive/evidgeo_modern'

## 1. Smoke test each model (2 prompts, with a trajectory). Look at the images before scaling up.

In [ ]:
from IPython.display import display
from PIL import Image
import glob
for m in ['sd35-medium', 'flux2-klein-base']:   # add 'qwen-image' on an 80 GB GPU
    out = f'/content/smoke_{m}'
    !python scripts/generate_modern.py --model {m} --master-manifest {MASTER} --out {out} --n-prompts 2 --seeds 1 --sweep 1 4 --traj-prompts 1 --traj-every 5
    for p in sorted(glob.glob(f'{out}/images/*/*/*.png'))[:6] + sorted(glob.glob(f'{out}/traj/*/cfg4/*.png')):
        print(p); display(Image.open(p).resize((192, 192)))

## 2. COCO captions: sweep × generic × i2i, 4 seeds, trajectories for 30 prompts
Captions are master rows 0–299, so every image pairs with its real COCO photo and with v2 prompts 0–299.
Roughly 300 × 8 conditions × 4 seeds ≈ 9.6k images per model. Resumable: re-run after a disconnect.

In [ ]:
# SD3.5-Medium's reduced design runs on Kaggle (dataset kaggle_sd35m); Colab runs the full design for these:
RUNS = [
    ('flux2-klein-base', ''),            # ~13 GB, fits a 40 GB A100 directly
    ('qwen-image', '--offload'),         # ~60 GB with text encoder: offload on a 40 GB A100 (needs High-RAM runtime)
]
for MODEL, extra in RUNS:
    !python scripts/generate_modern.py --model {MODEL} --master-manifest {MASTER} --out {BASE}/coco --n-prompts 300 --seeds 4 --traj-prompts 30 --traj-every 4 {extra}

## 3. GenEval 2 prompts (for failure prediction)
800 prompts with VQA atoms (CC BY-NC 4.0). Generated without i2i, at three guidance levels × 4 seeds.

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/facebookresearch/GenEval2/main/geneval2_data.jsonl -O /content/geneval2_data.jsonl
import json
rows = [json.loads(l) for l in open('/content/geneval2_data.jsonl')]
with open('/content/geneval2_manifest.jsonl', 'w') as f:
    for i, r in enumerate(rows):
        f.write(json.dumps({'global_id': i, 'caption': r['prompt']}) + '\n')
print(len(rows), 'prompts')
MODEL = 'sd35-medium'
!python scripts/generate_modern.py --model {MODEL} --master-manifest {MASTER} --prompts-jsonl /content/geneval2_manifest.jsonl --out {BASE}/geneval2 --n-prompts 800 --seeds 4 --modes sweep --sweep 1 4.5 9 --traj-prompts 100 --traj-every 4

### Score with the official GenEval 2 evaluator (Qwen3-VL-8B, Soft-TIFA)
It pins `transformers==4.57.0`, so it runs in its own virtualenv. One evaluator call per (model, mode, seed), because its image map is `{prompt: path}`.
**Check the first output JSON by eye.** The parser below expects a per-prompt score and stops with a clear error if the format differs.

In [ ]:
!git clone -q https://github.com/facebookresearch/GenEval2.git /content/GenEval2 2>/dev/null || true
!python -m venv /content/ge2 && /content/ge2/bin/pip -q install transformers==4.57.0 torch accelerate qwen-vl-utils pillow
import itertools, csv, os, json
G = f'{BASE}/geneval2'
prompts = {i: r['prompt'] for i, r in enumerate(rows)}
out_rows = []
for model in os.listdir(f'{G}/images'):
    for mode in os.listdir(f'{G}/images/{model}'):
        for seed in range(4):
            mp = {prompts[i]: f'{G}/images/{model}/{mode}/{i:05d}_seed{seed}.png' for i in prompts
                  if os.path.exists(f'{G}/images/{model}/{mode}/{i:05d}_seed{seed}.png')}
            mfile, ofile = f'/content/map_{model}_{mode}_{seed}.json', f'{G}/ge2_{model}_{mode}_{seed}.json'
            json.dump(mp, open(mfile, 'w'))
            if not os.path.exists(ofile):
                !cd /content/GenEval2 && /content/ge2/bin/python evaluation.py --benchmark_data /content/geneval2_data.jsonl --image_filepath_data {mfile} --method soft_tifa_gm --output_file {ofile}
            res = json.load(open(ofile))
            per = res.get('per_prompt', res) if isinstance(res, dict) else res
            assert isinstance(per, dict), f'unexpected GenEval2 output format in {ofile}: inspect and adapt'
            inv = {p: i for i, p in prompts.items()}
            for p, sc in per.items():
                sc = sc['score'] if isinstance(sc, dict) else sc
                if p in inv:
                    out_rows.append(dict(dataset='geneval2', model=model, mode=mode, gid=inv[p], seed=seed,
                                         score=float(sc), correct=int(float(sc) >= 0.5)))
with open(f'{G}/correctness.csv', 'w') as f:
    w = csv.DictWriter(f, fieldnames=list(out_rows[0])); w.writeheader(); w.writerows(out_rows)
print(len(out_rows), 'scored images')

## 4. Faces and memes (external sets)
Download the data yourself under its licence (research-only, no redistribution), then convert. `--text` sets which text the evidence is measured against. For memes this matters: OCR text (CLIP reads overlaid text), a literal caption, or the meme's meaning.

In [ ]:
EXT = '/content/drive/MyDrive/evidgeo_external'
!python scripts/prepare_external.py facecaption --n 3000 --out {EXT}/facecaption.jsonl
# !python scripts/prepare_external.py hateful_memes --src /content/drive/MyDrive/data/hateful_memes --out {EXT}/hateful_memes.jsonl
# !python scripts/prepare_external.py memecap --src /content/drive/MyDrive/data/memecap --text interpretation --out {EXT}/memecap_interp.jsonl
# !python scripts/prepare_external.py memecap --src /content/drive/MyDrive/data/memecap --text ocr --out {EXT}/memecap_ocr.jsonl

## 5. Evidence extraction + analysis (uncertainty, sweep trends, failure prediction, external labels)

In [ ]:
E = '/content/drive/MyDrive/evidgeo_eval'
!python scripts/run_eval.py --guidance-roots {BASE}/coco --trajectory-roots {BASE}/coco --master-manifest {MASTER} --real --out {E}/modern_coco --faith-per-bucket 100
!python scripts/run_eval.py --guidance-roots {BASE}/geneval2 --trajectory-roots {BASE}/geneval2 --out {E}/modern_geneval2
!python scripts/run_eval.py --external {EXT}/facecaption.jsonl --out {E}/external
!python scripts/analyze.py --run-dir {E}/modern_coco
!python scripts/analyze.py --run-dir {E}/modern_geneval2 --correctness {BASE}/geneval2/correctness.csv
!python scripts/analyze.py --run-dir {E}/external
import pandas as pd
for f in ['guidance_sweep', 'uncertainty_seed_by_mode', 'uncertainty_trajectory_by_mode', 'real_vs_generated']:
    p = f'{E}/modern_coco/results/{f}.csv'
    if os.path.exists(p): print(f); display(pd.read_csv(p).round(4))
display(pd.read_csv(f'{E}/modern_geneval2/results/failure_prediction.csv').sort_values('auroc', ascending=False).round(3))